<a href="https://colab.research.google.com/github/ZhafIrfan/ai-engineering-beware/blob/main/Beware.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import pandas as pd

df = pd.read_csv("returns_sustainability_dataset.csv")

print("=== 1. DATASET OVERVIEW & SAMPLE ===")
print("Total rows & columns:", df.shape)
sample = pd.concat([
    df[df['Return_Status'] == 'Returned'].head(8),
    df[df['Return_Status'] == 'Not Returned'].head(7)
])
print(sample[['Order_ID', 'Product_Category', 'Return_Status', 'Return_Reason', 'Days_to_Return', 'Profit_Loss']].to_string(index=False))

print("\n=== 2. COMPLETENESS & UNIQUENESS ===")
print("Missing values per column:\n", df.isna().sum())
print("Duplicate Order_IDs:", df['Order_ID'].duplicated().sum())
print("Exact duplicate rows:", df.duplicated().sum())

print("\n=== 3. VALIDATION RULE CHECKS ===")
# Rule 1: Logical consistency between Return_Status and Days_to_Return (Return_Date is missing from this dataset)
rule1_fail = df[
    ((df['Return_Status'] == 'Not Returned') & (df['Days_to_Return'] > 0)) |
    ((df['Return_Status'] == 'Returned') & (df['Days_to_Return'] <= 0))
]
print(f"Rule 1 Failures (Return Status vs Days mismatch): {len(rule1_fail)}")

# Rule 2: Non-negative pricing, return cost, and sustainability metrics
rule2_fail = df[
    (df['Product_Price'] <= 0) | (df['Order_Quantity'] <= 0) |
    (df['Return_Cost'] < 0) | (df['CO2_Saved'] < 0) | (df['Waste_Avoided'] < 0)
]
print(f"Rule 2 Failures (Invalid price, cost, or CO2/Waste values): {len(rule2_fail)}")

print("\n=== 4. COVERAGE & BIAS DISTRIBUTION ===")
print("Return Status Breakdown (%):\n", (df['Return_Status'].value_counts(normalize=True) * 100).round(2))
print("\nProduct Category Breakdown (%):\n", (df['Product_Category'].value_counts(normalize=True) * 100).round(2))
print("\nReturn Rate by Gender (%):\n", (df.groupby('User_Gender')['Return_Status'].apply(lambda x: (x == 'Returned').mean() * 100)).round(2))
print("\nDate Range:", df['Order_Date'].min(), "to", df['Order_Date'].max())

=== 1. DATASET OVERVIEW & SAMPLE ===
Total rows & columns: (5000, 23)
Order_ID Product_Category Return_Status Return_Reason  Days_to_Return  Profit_Loss
ORD00001             Toys      Returned    Size Issue              12  2418.347140
ORD00004  Home Appliances      Returned    Size Issue              11  4721.525630
ORD00006         Clothing      Returned    Size Issue              48   651.993016
ORD00009         Clothing      Returned  Changed Mind              41   395.410020
ORD00011         Clothing      Returned    Wrong Item              27  4501.088280
ORD00012         Clothing      Returned    Size Issue              29  1899.458725
ORD00016            Books      Returned    Wrong Item              28  1364.957296
ORD00021             Toys      Returned     Defective              38  1410.088840
ORD00000         Clothing  Not Returned     No Return               0  2393.163468
ORD00002         Clothing  Not Returned     No Return               0  2567.404800
ORD00003         